#### Aspect Keywords from Literature

In [ ]:
staff_keywords = [

    "staff",
    "service",
    "employee",
    "employees",
    "personnel",
    "team",
    "worker"

    "reception",
    "receptionist",
    "front desk",
    "check in",
    "check-in",

    "manager",
    "host",
    "hostess",
    "concierge",

    "friendly",
    "helpful",
    "welcoming",
    "professional",
    "courteous",

    "customer service",
    "hospitality"
]

In [ ]:
cleanliness_keywords = [

    "clean",
    "cleanliness",
    "spotless",
    "tidy",

    "dirty",
    "messy",
    "dust",
    "dusty",

    "hygiene",
    "sanitary",

    "bathroom",
    "toilet",
    "shower",

    "bed",
    "bedsheets",
    "linen",
    "towel",

    "stain",
    "smell",
    "odour",
    "odor"
]

In [ ]:
location_keywords = [

    "location",
    "distance",

    "city centre",
    "city center",
    "downtown",
    "city",
    "center",
    "centre",

    "airport",
    "station",
    "bus",

    "walking distance",
    "nearby",

    "beach",
    "sea",
    "view",
    "lake",
    "pool",
    "ocean",
    "waterfall",

    "transport",
    "access",

    "central",
    "convenient",

    "parking",
    "park",
    "mall",
    "shopping",
]

In [ ]:
value_keywords = [

    "value",
    "value for money",

    "price",
    "pricing",
    "cost",

    "expensive",
    "cheap",

    "affordable",
    "budget",

    "worth",
    "worthwhile",

    "overpriced",
    "reasonable",

    "money",
    "paid",
    "rate"
]

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd

processed_path = "/content/drive/MyDrive/Colab Notebooks/Irish Hotel Reviews/Dataset/processed/irish_hotel_reviews_processed.csv"

df = pd.read_csv(processed_path)

print(df.shape)
print(df.columns)

(13294, 13)
Index(['review_id', 'review_text', 'overall_rating', 'hotel_name', 'county',
       'trip_type', 'language', 'clean_review', 'word_count', 'sentiment',
       'anonymized_review', 'final_review', 'clean_review_ml'],
      dtype='object')


#### Generate candidate keywords using TF-IDF

In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd

vectorizer = TfidfVectorizer(
    max_features=1000,
    stop_words='english',
    ngram_range=(1,2)
)

X = vectorizer.fit_transform(df["clean_review_ml"])

terms = vectorizer.get_feature_names_out()

tfidf_scores = X.sum(axis=0).A1

tfidf_df = pd.DataFrame({
    "term": terms,
    "score": tfidf_scores
})

tfidf_df = tfidf_df.sort_values(
    by="score",
    ascending=False
)

print(tfidf_df.head(100))

       term       score
416   hotel  825.506130
623  person  788.579449
719    room  680.098653
804   staff  665.199572
366   great  620.997009
..      ...         ...
825    star  114.029365
559  minute  113.818496
940    view  113.793169
204   didnt  113.158666
155    come  112.427519

[100 rows x 2 columns]


In [4]:
tfidf_df.to_csv(
    "/content/drive/MyDrive/Colab Notebooks/Irish Hotel Reviews/Dataset/candidate_keywords_tfidf.csv",
    index=False
)

#### Build Word2Vec Model

In [5]:
pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 62.5 MB/s eta 0:00:00


In [6]:
# train the model
from gensim.models import Word2Vec

sentences = [
    review.split()
    for review in df["clean_review_ml"]
]

model = Word2Vec(
    sentences,
    vector_size=100,
    window=5,
    min_count=5,
    workers=4
)

In [7]:
model.save("/content/drive/MyDrive/Colab Notebooks/Irish Hotel Reviews/Dataset/word2vec_model/hotel_reviews_word2vec.model")

In [ ]:
model.wv.most_similar(
    "clean",
    topn=20
)

[('spotless', 0.9123284220695496),
 ('tidy', 0.8998678922653198),
 ('spacious', 0.8957071304321289),
 ('spotlessly', 0.878776490688324),
 ('modern', 0.8659928441047668),
 ('appointed', 0.8477475047111511),
 ('cozy', 0.839678168296814),
 ('nicely', 0.8346102833747864),
 ('bright', 0.8318159580230713),
 ('quiet', 0.82569420337677),
 ('immaculate', 0.8215550780296326),
 ('stylish', 0.8133358359336853),
 ('equipped', 0.8084845542907715),
 ('cosy', 0.8077245950698853),
 ('decor', 0.7996659278869629),
 ('decorated', 0.7940950989723206),
 ('amenity', 0.7867333889007568),
 ('maintained', 0.7847363948822021),
 ('elegantly', 0.7746566534042358),
 ('comfy', 0.7720158100128174)]

#### Seed words

##### Staff

In [8]:
staff_seed_words = [
    "staff",
    "service",
    "reception"
]

In [9]:
cleanliness_seed_words = [
    "clean",
    "cleanliness",
    "dirty"
]

In [10]:
location_seed_words = [
    "location",
    "airport",
    "centre"
]

In [11]:
value_seed_words = [
    "value",
    "price",
    "cost"
]

In [12]:
import pandas as pd

staff_results = []

for word in staff_seed_words:
    results = model.wv.most_similar(word, topn=20)

    for discovered_word, similarity in results:
        staff_results.append({
            "aspect":"staff",
            "seed_word": word,
            "discovered_word": discovered_word,
            "similarity": similarity
        })

staff_df = pd.DataFrame(staff_results)

print(staff_df)

   aspect  seed_word discovered_word  similarity
0   staff      staff          polite    0.678115
1   staff      staff       efficient    0.671238
2   staff      staff      incredibly    0.667533
3   staff      staff           super    0.663394
4   staff      staff       extremely    0.662014
5   staff      staff    approachable    0.657118
6   staff      staff        employee    0.651011
7   staff      staff   knowledgeable    0.649639
8   staff      staff    professional    0.646574
9   staff      staff            kind    0.627297
10  staff      staff     encountered    0.620617
11  staff      staff          chatty    0.618939
12  staff      staff       courteous    0.614193
13  staff      staff          tierna    0.612929
14  staff      staff        everyone    0.611568
15  staff      staff   exceptionally    0.607303
16  staff      staff          smiley    0.603103
17  staff      staff           kayla    0.601852
18  staff      staff   accommodating    0.594285
19  staff      staff

In [13]:
import pandas as pd

cleanliness_results = []

for word in cleanliness_seed_words:
    results = model.wv.most_similar(word, topn=20)

    for discovered_word, similarity in results:
        cleanliness_results.append({
            "aspect":"cleanliness",
            "seed_word": word,
            "discovered_word": discovered_word,
            "similarity": similarity
        })

cleanliness_df = pd.DataFrame(cleanliness_results)

print(cleanliness_df)

         aspect    seed_word discovered_word  similarity
0   cleanliness        clean        spotless    0.911565
1   cleanliness        clean        spacious    0.897572
2   cleanliness        clean            tidy    0.888069
3   cleanliness        clean      spotlessly    0.864737
4   cleanliness        clean            cozy    0.843674
5   cleanliness        clean          modern    0.842861
6   cleanliness        clean       appointed    0.840106
7   cleanliness        clean          nicely    0.836209
8   cleanliness        clean          bright    0.832193
9   cleanliness        clean           quiet    0.830904
10  cleanliness        clean         stylish    0.822626
11  cleanliness        clean         amenity    0.804656
12  cleanliness        clean            cosy    0.802520
13  cleanliness        clean        equipped    0.802387
14  cleanliness        clean      immaculate    0.798978
15  cleanliness        clean           decor    0.788368
16  cleanliness        clean   

In [ ]:
import pandas as pd

cleanliness_results = []

for word in cleanliness_seed_words:
    results = model.wv.most_similar(word, topn=20)

    for discovered_word, similarity in results:
        cleanliness_results.append({
            "aspect":"cleanliness",
            "seed_word": word,
            "discovered_word": discovered_word,
            "similarity": similarity
        })

cleanliness_df = pd.DataFrame(cleanliness_results)

print(cleanliness_df)

In [14]:
import pandas as pd

location_results = []

for word in location_seed_words:
    results = model.wv.most_similar(word, topn=20)

    for discovered_word, similarity in results:
        location_results.append({
            "aspect":"location",
            "seed_word": word,
            "discovered_word": discovered_word,
            "similarity": similarity
        })

location_df = pd.DataFrame(location_results)

print(location_df)

      aspect seed_word discovered_word  similarity
0   location  location         located    0.777168
1   location  location        situated    0.726710
2   location  location            base    0.726078
3   location  location       exploring    0.713897
4   location  location            spot    0.693984
5   location  location          center    0.689180
6   location  location      attraction    0.686769
7   location  location         central    0.685736
8   location  location          centre    0.685618
9   location  location       centrally    0.678355
10  location  location            city    0.672727
11  location  location          dublin    0.671776
12  location  location      positioned    0.671025
13  location  location           arena    0.670731
14  location  location           close    0.667825
15  location  location      everywhere    0.667518
16  location  location         explore    0.667228
17  location  location        shopping    0.665303
18  location  location     walk

In [15]:
import pandas as pd

value_results = []

for word in value_seed_words:
    results = model.wv.most_similar(word, topn=20)

    for discovered_word, similarity in results:
        value_results.append({
            "aspect":"value",
            "seed_word": word,
            "discovered_word": discovered_word,
            "similarity": similarity
        })

value_df = pd.DataFrame(value_results)

print(value_df)

   aspect seed_word discovered_word  similarity
0   value     value           money    0.823748
1   value     value           price    0.745012
2   value     value      reasonable    0.724067
3   value     value         overall    0.709575
4   value     value            rate    0.698880
5   value     value          priced    0.665071
6   value     value         quality    0.650663
7   value     value         amenity    0.615317
8   value     value          choice    0.612449
9   value     value      reasonably    0.594395
10  value     value        location    0.593475
11  value     value           range    0.584566
12  value     value     considering    0.582874
13  value     value           picky    0.578196
14  value     value        facility    0.577530
15  value     value     cleanliness    0.574285
16  value     value     welllocated    0.566362
17  value     value         variety    0.563179
18  value     value            size    0.560071
19  value     value     environment    0

In [16]:
combined_df = pd.concat(
    [staff_df, cleanliness_df, location_df, value_df],
    axis=0,          # stack rows (default)
    ignore_index=True
)

print(combined_df.head())

  aspect seed_word discovered_word  similarity
0  staff     staff          polite    0.678115
1  staff     staff       efficient    0.671238
2  staff     staff      incredibly    0.667533
3  staff     staff           super    0.663394
4  staff     staff       extremely    0.662014


In [18]:
combined_df.shape

(240, 4)

In [19]:
combined_df = (
    combined_df
    .sort_values("similarity", ascending=False)
    .drop_duplicates(subset="discovered_word")
    .reset_index(drop=True)
)

In [20]:
combined_df.head()

,aspect,seed_word,discovered_word,similarity
0,cleanliness,dirty,fan,0.970143
1,cleanliness,dirty,hair,0.961676
2,cleanliness,dirty,toilet,0.957147
3,value,cost,website,0.956843
4,cleanliness,dirty,tap,0.956258


In [21]:
combined_df.to_csv(
    "aspect_expansion_table.csv",
    index=False
)

In [ ]:
staff_word2vec_keywords = [
    "receptionist",
    "gentleman",
    "doorman",
    "young",
    "man",
    "concierge",
    "checking",
    "lady",
    "dealt",
    "greeted",
    "chatty",
    "met",
    "kind",
    "greeting",
    "courteous",
    "checkin",
    "server",
    "polite",
    "efficient",
    "incredibly",
    "alberto",
    "extremely",
    "approachable",
    "waiter",
    "employee",
    "knowledgeable",
    "professional",
    "consistently",
    "attention",
    "presentation",
    "particular",
    "encountered",
    "job",
    "everyone",
    "exceptionally",
    "bartender",
    "hospitality",
    "smiley",
    "prompt",
    "cleanliness",
    "host",
    "accommodating",
    "top",
    "customer",
    "team",
    "personable"
]

In [ ]:
cleanliness_word2vec_keywords = [
    "fan",
    "hair",
    "toilet",
    "tap",
    "light",
    "heat",
    "wash",
    "broken",
    "carpet",
    "mirror",
    "sink",
    "clothes",
    "properly",
    "ac",
    "curtain",
    "spotless",
    "spacious",
    "tidy",
    "comfort",
    "spotlessly",
    "high",
    "quality",
    "cozy",
    "modern",
    "bright",
    "standard",
    "amenity",
    "cosy",
    "equipped",
    "immaculate",
    "decor",
    "maintained",
    "elegant",
    "comfy",
    "luxurious",
    "interior",
    "impressive",
    "accommodation",
    "importantly",
    "wellappointed",
    "aesthetically",
]

In [ ]:
location_word2vec_keywords = [
    "center",
    "shuttle",
    "bus",
    "stop",
    "dart",
    "hop",
    "exploring",
    "tram",
    "train",
    "handy",
    "right",
    "positioned",
    "ride",
    "transport",
    "across",
    "downtown",
    "explore",
    "transportation",
    "sight",
    "road",
    "arena",
    "close",
    "walkable",
    "reach",
    "near",
    "situated",
    "town",
    "central",
    "heart",
    "attraction",
    "stroll",
    "stadium",
    "shopping",
    "convenient",
    "everywhere",
    "located",
    "base",
    "spot",
    "centre",
    "centrally",
    "city",
    "walkability",
]

In [ ]:
value_word2vec_keywords = [
    "avoid",
    "paying",
    "laundry",
    "tired",
    "slightly",
    "least",
    "worried",
    "vent",
    "reasonable",
    "expensive",
    "money",
    "rate",
    "considering",
    "expect",
    "offer",
    "price",
    "value",
    "expected",
    "fine",
    "decent",
    "priced",
    "overall",
    "outrageous",
    "compared",
    "average",
    "pricey",
    "reasonably",
    "choice",
    "range",
    "picky",
    "facility",
    "welllocated",
    "variety",
    "size"
]